# From Scratch GAN on MNIST (Pytorch)

**abstract:** Here a Generative adversarial network is trained on MNIST dataset using DCGAN architecture adapted to 28x28 grayscale images. The discriminator is a binary classifier trained to distinguish real images from fake images, while the generator learns to synthesize the realistic digits from random noise. 

$$
\begin{align*}
& \textbf{1. GAN Objective (Theory)} \\
& \text{A vanilla GAN consists of a \textbf{generator} } G_\theta \text{ that maps latent noise } z \sim p_z(z) \text{ to images } G(z), \\
& \text{and a \textbf{discriminator} } D_\phi \text{ that outputs the probability that an input image is real. The original minimax objective is:} \\
& \min_{\theta}\max_{\phi}\; V(D_\phi,G_\theta) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D_\phi(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D_\phi(G_\theta(z)))]. \\
\\
& \text{In practice, the \textbf{non-saturating} generator loss is used to provide stronger gradients:} \\
& \mathcal{L}_D = -\mathbb{E}_{x \sim p_{\text{data}}}[\log D_\phi(x)] - \mathbb{E}_{z \sim p_z}[\log(1 - D_\phi(G_\theta(z)))], \\
& \mathcal{L}_G = -\mathbb{E}_{z \sim p_z}[\log D_\phi(G_\theta(z))]. \\
\\
& \text{I implement these with \textbf{binary cross-entropy with logits} (BCEWithLogitsLoss), letting the discriminator} \\
& \text{output raw logits } s \text{ where } D(x) = \sigma(s) \text{ and } \sigma(\cdot) \text{ is the sigmoid function.} \\
\\
& \textbf{2. DCGAN Heuristics (adapted to 28×28 MNIST)} \\
& \text{- \textbf{Generator:} latent } z \in \mathbb{R}^{100} \to \text{linear} \to \text{reshape to } 7 \times 7 \text{ feature map} \to \text{transposed convolutions to } 14 \times 14 \\
& \quad \text{and } 28 \times 28 \to \text{final } 1 \text{ channel with \textbf{tanh}.} \\
& \text{- \textbf{Discriminator:} strided convolutions } 28 \to 14 \to 7, \text{ LeakyReLU activations, batch norm in deeper layers, and a} \\
& \quad \text{single-logit output (real vs fake).} \\
& \text{- \textbf{Optimizer:} Adam with } \beta_1=0.5, \beta_2=0.999. \\
& \text{- \textbf{Input normalization:} images scaled to } [-1,1] \text{ to match generator \textbf{tanh} output.} \\
\\
& \textbf{3. Labels and Stabilization} \\
& \text{I use target labels } 1 \text{ for real and } 0 \text{ for fake. Optional tricks like one-sided label smoothing or random label} \\
& \text{flips can be added later to improve stability.}
\end{align*}
$$

# Import the necessary libraries

In [ ]:
import os, math, random
import numpy as np
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms, utils
import matplotlib.pyplot as plt

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
OUT_DIR = "/kaggle/working/gan_mnist"
SAMPLES_DIR = os.path.join(OUT_DIR, "samples")
os.makedirs(SAMPLES_DIR, exist_ok=True)


# MNIST dataloader

In [ ]:
BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor(),                # -> [0,1]
    transforms.Normalize((0.5,), (0.5,))  # -> [-1,1] for GAN (tanh output)
])

trainset = datasets.MNIST(
    root="./data",   # will download here
    train=True,
    download=True,
    transform=transform
)

trainloader = DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Total samples:", len(trainset))
print("Batches per epoch:", len(trainloader))

# models (Generator and Discriminators)

In [ ]:
LATENT_DIM=100

class Generator(nn.Module):
    def __init__(self,z_dim=LATENT_DIM,base=64):
        super().__init__()
        self.fc=nn.Sequential(
            nn.Linear(z_dim,256*7*7),
            nn.BatchNorm1d(256*7*7),
            nn.ReLU(True)
        )
        self.deconv=nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # 7->14
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # 14->28
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
    def forward(self,z):
        x=self.fc(z)
        x=x.view(z.size(0),256,7,7)
        x=self.deconv(x)
        return x

In [ ]:
class Discriminator(nn.Module):
    # (1,28,28) -> logit
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),   # 28->14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # 14->7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(128*7*7, 1)  

    def forward(self, x):
        h = self.conv(x)                 # (N, 128, 7, 7)
        h = h.view(x.size(0), -1)        # (N, 128*7*7)
        logit = self.fc(h)               # (N, 1)
        return logit


In [ ]:
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

print("Generator params:", sum(p.numel() for p in G.parameters())/1e6, "M")
print("Discriminator params:", sum(p.numel() for p in D.parameters())/1e6, "M")

# Loss and optimizers

In [ ]:
lr = 2e-4
G_OPT = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
D_OPT = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
criterion = nn.BCEWithLogitsLoss()

In [ ]:
FIXED_Z = torch.randn(64, LATENT_DIM, device=DEVICE)

In [ ]:
def save_grid(tensor, fname, nrow=8):
    grid = utils.make_grid(tensor, nrow=nrow, normalize=True, value_range=(-1, 1))
    utils.save_image(grid, fname)

$$
\begin{align*}
& \textbf{4. Training Losses (with BCE logits)} \\
\\
& \text{Let } s_r = D_\phi(x) \text{ and } s_f = D_\phi(G_\theta(z)) \text{ denote discriminator logits.} \\
& \text{Using \textbf{BCEWithLogitsLoss}, we minimize:} \\
\\
& \mathcal{L}_D = \mathrm{BCEWithLogits}(s_r, 1) + \mathrm{BCEWithLogits}(s_f, 0), \\
& \mathcal{L}_G = \mathrm{BCEWithLogits}(s_f, 1). \\
\\
& \text{These correspond to the non-saturating GAN formulation and yield stronger} \\
& \text{generator gradients early in training.}
\end{align*}
$$

# Training Loop

In [ ]:
EPOCHS = 200
log_every = 200

G_losses, D_losses = [], []

for epoch in range(1, EPOCHS+1):
    G.train(); D.train()
    for i, (x_real, _) in enumerate(trainloader, start=1):
        x_real = x_real.to(DEVICE)  
        bs = x_real.size(0)

        z = torch.randn(bs, LATENT_DIM, device=DEVICE)
        x_fake = G(z).detach()

        D_OPT.zero_grad(set_to_none=True)


        real_logits = D(x_real)
        real_labels = torch.ones(bs, 1, device=DEVICE)
        d_real = criterion(real_logits, real_labels)

        fake_logits = D(x_fake)
        fake_labels = torch.zeros(bs, 1, device=DEVICE)
        d_fake = criterion(fake_logits, fake_labels)

        d_loss = d_real + d_fake
        d_loss.backward()
        D_OPT.step()

        z = torch.randn(bs, LATENT_DIM, device=DEVICE)
        x_fake = G(z)
        G_OPT.zero_grad(set_to_none=True)

        fake_logits = D(x_fake)
        g_loss = criterion(fake_logits, real_labels)
        g_loss.backward()
        G_OPT.step()

        # Logging
        if i % log_every == 0:
            print(f"[Epoch {epoch:02d}/{EPOCHS}] Step {i:04d} | D: {d_loss.item():.3f} | G: {g_loss.item():.3f}")

    with torch.no_grad():
        samples = G(FIXED_Z).cpu()
    save_grid(samples, os.path.join(SAMPLES_DIR, f"epoch_{epoch:03d}.png"), nrow=8)

    G_losses.append(g_loss.item())
    D_losses.append(d_loss.item())

print("Training complete. Samples saved to:", SAMPLES_DIR)

# Visualizations

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(G_losses, label="G loss")
plt.plot(D_losses, label="D loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("GAN Training Losses")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "loss_curves.png"), dpi=150)
plt.show()

In [ ]:
from IPython.display import Image, display
latest = os.path.join(SAMPLES_DIR, sorted(os.listdir(SAMPLES_DIR))[-1])
display(Image(latest))

$$
\begin{align*}
& \textbf{5. Results \& Notes} \\
\\
& \text{- \textbf{Qualitative Samples.} The grids saved at \texttt{samples/epoch\_XXX.png} show clear progression} \\
& \quad \text{from noisy blobs to digit-like structures.} \\
\\
& \text{- \textbf{Loss Behavior.} The discriminator loss } \mathcal{L}_D \text{ and generator loss } \mathcal{L}_G \text{ may oscillate.} \\
& \quad \text{This is expected due to the adversarial nature of training.} \\
\\
& \text{- \textbf{Stability Tips.}} \\
& \quad \text{- Keep } \beta_1=0.5 \text{ for Adam (DCGAN recommendation).} \\
& \quad \text{- Normalize inputs to } [-1,1] \text{ and use \textbf{tanh} at the generator output.} \\
& \quad \text{- If collapse occurs, try smaller learning rate, add one-sided label smoothing} \\
& \qquad \text{(e.g., real label } =0.9\text{), or add instance noise.}
\end{align*}
$$

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=DEVICE)
    imgs = G(z).cpu()
save_grid(imgs, os.path.join(SAMPLES_DIR, "inference_grid.png"), nrow=8)
from IPython.display import Image, display
display(Image(os.path.join(SAMPLES_DIR, "inference_grid.png")))